In [1]:
import os
import pandas as pd
import numpy as np
try:
    import xmltodict
except:
    ! pip install xmltodict
    
from parse_payload import PayloadToDataFrame
import time
from tqdm import tqdm
import boto3

In [2]:
# download from s3
def download_from_s3(str_local_path, str_bucket_path, str_project):
    # init client
    cls_client = boto3.client(
        's3',
    )
    # download file
    cls_client.download_file(
        str_project, 
        str_bucket_path, 
        str_local_path,
    )

In [3]:
# constants
str_project = '20241112-simple-model-test'
str_task = '01_parse_payloads'
str_datecol = 'dtmCreatedDate'
str_filename = 'df_requests_2022-10-14.gzip' # edit as needed

In [4]:
%%time

# read file
print('Importing file...')
str_uri = f's3://20241022-parse-snowflake-payloads/deprecated/03_pull_payloads_tbldove/{str_filename}'
df = pd.read_parquet(str_uri)
df

Importing file...


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fsspec/registry.py:279: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


CPU times: user 28.4 s, sys: 16.7 s, total: 45.2 s
Wall time: 51.6 s


,bigDoveId,bigAccountId,strRequest,dtmCreatedDate
0,1879245,6383567,"{""request_id"":""6383567252403"",""rows"":[{""row_id...",2022-10-14 00:01:22.2951870
1,1879246,6383569,"{""request_id"":""6383569756746"",""rows"":[{""row_id...",2022-10-14 00:03:35.6596965
2,1879247,6383451,"{""request_id"":""6383451743193"",""rows"":[{""row_id...",2022-10-14 00:03:41.7782127
3,1879248,6383570,"{""request_id"":""6383570645917"",""rows"":[{""row_id...",2022-10-14 00:03:46.6198857
4,1879249,6383571,"{""request_id"":""6383571961538"",""rows"":[{""row_id...",2022-10-14 00:03:50.4222707
...,...,...,...,...
42561,1921806,5629175,"{""request_id"":""5629175628166"",""rows"":[{""row_id...",2022-10-14 23:59:53.1357990
42562,1921807,5629180,"{""request_id"":""5629180956085"",""rows"":[{""row_id...",2022-10-14 23:59:56.3314356
42563,1921808,5629195,"{""request_id"":""5629195452609"",""rows"":[{""row_id...",2022-10-14 23:59:56.8363077
42564,1921809,5629185,"{""request_id"":""5629185521181"",""rows"":[{""row_id...",2022-10-14 23:59:57.6128286


In [5]:
# init
cls_parse_payload = PayloadToDataFrame()

In [6]:
# iterate through rows and extract raw data
list_df_tmp = []
for a, str_request in enumerate(tqdm(df['strRequest'])):
    # replace NaN
    time_start = time.perf_counter()

    # get data
    df_tmp = cls_parse_payload.get_data(str_request=str_request)

    # make copy
    df_tmp = df_tmp.copy()

    # assign
    df_tmp['BIGDOVEID'] = df['bigDoveId'].iloc[a]
    df_tmp['ACCOUNTID'] = df['bigAccountId'].iloc[a]
    df_tmp['REQUEST_DATETIME'] = df['dtmCreatedDate'].iloc[a]
    
    # get number of rows
    int_nrows = df_tmp.shape[0]
    
    # logic
    if int_nrows == 1:
        list_bitdebtor = [1]
    else:
        list_bitdebtor = [1, 0]
    
    # debtor
    df_tmp['BITDEBTOR'] = list_bitdebtor
    
    # reorder
    list_cols_id = [
        'BIGDOVEID',
        'ACCOUNTID',
        'REQUEST_DATETIME',
        'BITDEBTOR',
    ]
    list_cols = [col for col in df_tmp.columns if col not in list_cols_id]
    list_cols = list_cols_id + list_cols
    df_tmp = df_tmp[list_cols].copy()

    # end time
    time_end = time.perf_counter()
    # sec
    flt_sec = time_end - time_start

    # assign
    df_tmp['flt_sec'] = flt_sec
    
    # append
    list_df_tmp.append(df_tmp)

100%|██████████| 42566/42566 [1:30:53<00:00,  7.80it/s]  


In [7]:
%%time

# make df
df = pd.concat(list_df_tmp)
# save memory
del list_df_tmp
# show
df

CPU times: user 32min 31s, sys: 37.5 s, total: 33min 8s
Wall time: 33min 7s


,BIGDOVEID,ACCOUNTID,REQUEST_DATETIME,BITDEBTOR,uniqueid__app,bigaccountid__app,bigdebtorid__app,bitdebtor__app,bigstatusid__app,strcity__app,...,linkc047__tu,linkc049__tu,linkc005__tu,linkc004__tu,linkc051__tu,linkc014__tu,linkc015__tu,linkc016__tu,linkc022__tu,linkc023__tu
0,1879245,6383567,2022-10-14 00:01:22.2951870,1,0__0__20221013,6383567,NaN,1,1,SEATTLE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,1879245,6383567,2022-10-14 00:01:22.2951870,0,0__0__20221013,6383567,NaN,0,1,SEATTLE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,1879246,6383569,2022-10-14 00:03:35.6596965,1,0__0__20221013,6383569,NaN,1,1,FORT LAUDERDALE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,1879246,6383569,2022-10-14 00:03:35.6596965,0,0__0__20221013,6383569,NaN,0,1,FORT LAUDERDALE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,1879247,6383451,2022-10-14 00:03:41.7782127,1,0__0__20221013,6383451,NaN,1,5,Lincoln,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,1921806,5629175,2022-10-14 23:59:53.1357990,1,0__0__20210416,5629175,NaN,1,14,CHARLOTTE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,1921807,5629180,2022-10-14 23:59:56.3314356,1,0__0__20210416,5629180,NaN,1,5,Thomaston,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
0,1921808,5629195,2022-10-14 23:59:56.8363077,1,0__0__20210416,5629195,NaN,1,5,MONTGOMERY,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
0,1921809,5629185,2022-10-14 23:59:57.6128286,1,0__0__20210416,5629185,NaN,1,3,BEALETON,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN


In [8]:
%%time

# convert date to datetime
df['REQUEST_DATETIME'] = pd.to_datetime(df['REQUEST_DATETIME'])

CPU times: user 11.2 ms, sys: 0 ns, total: 11.2 ms
Wall time: 10.9 ms


In [9]:
%%time

# identify the non-numeric columns
list_cols = []
for col in df.columns:
    str_dtype = df[col].dtype
    if str_dtype not in ['int64','float64']:
        list_cols.append(col)
    else:
        pass
int_n_cols = len(list_cols)
print(f'There are {int_n_cols} possible non-numeric columns')

There are 62 possible non-numeric columns
CPU times: user 43 ms, sys: 0 ns, total: 43 ms
Wall time: 42.5 ms


In [10]:
%%time

# remove dates
list_cols = [col for col in list_cols if col != 'REQUEST_DATETIME']

# convert non-numeric to string
for col in tqdm(list_cols):
    try:
        # convert to string
        df[col] = df[col].astype(str)
    except:
        pass

100%|██████████| 61/61 [00:03<00:00, 16.52it/s]

CPU times: user 3.02 s, sys: 682 ms, total: 3.7 s
Wall time: 3.69 s


In [11]:
%%time

# save
str_filename_date = str_filename.split('_')[2]
str_filename = f'df_parsed_{str_filename_date}'
str_uri = f's3://{str_project}/{str_task}/{str_filename}'
df.to_parquet(str_uri, compression='gzip')

CPU times: user 21 s, sys: 108 ms, total: 21.1 s
Wall time: 23.7 s
